## Model API

The `VCO_Model` class encapsulates all relevant relationships:

- `fosc_from_vin(v_in)` – compute oscillation frequency for a given input voltage
- `vin_from_fosc(f_osc)` – invert the model to extract input voltage
- `conductance(v_in, i_dc)` – calculate conductance from voltage and current
- `dG_df(v_in, i_dc)` – dG/d(f_osc) at a given operating point; quantifies G uncertainty due to f_osc measurement error
- `dG_df_curve(i_dc_range)` – matrix of G tolerances (dG/d(f_osc)) across a range of currents and voltages

You can instantiate the model with your measured datasets and then call these methods directly for analysis or testing, independent of the plotting routines.

# PoI Plotter

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

## VCO_Model class

In [2]:
# ---------------------------------------------------------------------------
# Data Models and Computation Classes
# ---------------------------------------------------------------------------

class VCO_Model:
    """
    Encapsulate f_osc ↔ V_in ↔ G relations and dG_df calculations.

    Piecewise model:
        f_osc(V_in) = 0                    if V_in < piecewise_threshold
                    = a*V_in^2 + b*V_in + c otherwise

    Notes:
    - V_in is in mV
    - f_osc is in kHz
    - i_dc is in uA
    - G is in uS
    """

    def __init__(self, vin_data, fosc_data, vdd=0.8):
        self.vin_data = np.asarray(vin_data, dtype=float)
        self.fosc_data = np.asarray(fosc_data, dtype=float)
        self.vdd = float(vdd)  # V

        nonzero_indices = np.where(self.fosc_data > 0)[0]
        if len(nonzero_indices) == 0:
            raise ValueError("fosc_data contains no positive values; cannot determine threshold.")

        # First Vin where oscillation starts
        self.piecewise_threshold = float(self.vin_data[nonzero_indices[0]])
        print(f"Determined piecewise threshold: {self.piecewise_threshold:.2f} mV")

        # Fit polynomial only in active region
        mask = self.vin_data >= self.piecewise_threshold
        vin_active = self.vin_data[mask]
        fosc_active = self.fosc_data[mask]

        self.popt_poly, _ = curve_fit(self.polynomial_model, vin_active, fosc_active)

    @staticmethod
    def polynomial_model(x, a, b, c):
        return a * x**2 + b * x + c

    def fosc_from_vin(self, vin_mV):
        """
        Piecewise polynomial model:
            f_osc = 0 if V_in < piecewise_threshold, else polynomial
        """
        vin_arr = np.asarray(vin_mV, dtype=float)
        a, b, c = self.popt_poly
        poly_val = np.maximum(a * vin_arr**2 + b * vin_arr + c, 0.0)
        out = np.where(vin_arr < self.piecewise_threshold, 0.0, poly_val)
        return out.item() if np.isscalar(vin_mV) else out

    def vin_from_fosc(self, fosc_kHz):
        """
        Invert the piecewise model to extract V_in from f_osc.
        For f_osc <= 0, returns the threshold as a lower bound.
        """
        fosc_arr = np.asarray(fosc_kHz, dtype=float)
        a, b, c = self.popt_poly

        out = np.full_like(fosc_arr, np.nan, dtype=float)

        # Below or at zero frequency -> below threshold / lower bound
        zero_mask = fosc_arr <= 0
        out[zero_mask] = self.piecewise_threshold

        # Invert quadratic only for positive fosc
        pos_mask = fosc_arr > 0
        if np.any(pos_mask):
            fpos = fosc_arr[pos_mask]
            disc = b**2 - 4 * a * (c - fpos)

            valid_disc = disc >= 0
            v_candidates = np.full_like(fpos, np.nan, dtype=float)

            # Two roots
            sqrt_disc = np.sqrt(np.maximum(disc, 0.0))
            v1 = (-b + sqrt_disc) / (2 * a)
            v2 = (-b - sqrt_disc) / (2 * a)

            # Keep the physically valid root above threshold
            cand1_ok = (v1 >= self.piecewise_threshold) & valid_disc
            cand2_ok = (v2 >= self.piecewise_threshold) & valid_disc

            v_candidates[cand1_ok] = v1[cand1_ok]

            replace_mask = np.isnan(v_candidates) & cand2_ok
            v_candidates[replace_mask] = v2[replace_mask]

            out[pos_mask] = v_candidates

        return out.item() if np.isscalar(fosc_kHz) else out

    def vdd_mV(self):
        return self.vdd * 1e3

    def conductance(self, vin_mV, i_dc_uA):
        """
        G in uS given V_in and i_dc.

        Uses:
            G = i_dc / (VDD - V_in)

        so vin_mV is assumed to be the sensed node voltage and the resistor
        drop is (VDD - V_in).
        """
        vin_arr = np.asarray(vin_mV, dtype=float)
        i_arr = np.asarray(i_dc_uA, dtype=float)

        vin_V = vin_arr * 1e-3
        denom = self.vdd - vin_V  # resistor voltage drop in V
        i_dc_A = i_arr * 1e-6

        # Broadcast-safe computation
        with np.errstate(divide='ignore', invalid='ignore'):
            G_uS = (i_dc_A / denom) * 1e6

        invalid = (vin_arr < self.piecewise_threshold) | np.isnan(vin_arr) | (denom <= 0)
        G_uS = np.where(invalid, np.nan, G_uS)

        return G_uS.item() if np.isscalar(vin_mV) and np.isscalar(i_dc_uA) else G_uS

    def dG_df(self, vin_mV, i_dc_uA):
        """
        G tolerance (dG/df) in units of uS/kHz.
        
        Quantifies how much G changes with f_osc measurement error.
        Higher value = greater dG_df to f_osc measurement uncertainty.

        Since:
            G = i_dc / (VDD - V_in)
        then:
            dG/dV = i_dc / (VDD - V_in)^2
        and:
            df/dV = 0 below threshold
                    2*a*V + b above threshold
        """
        vin_arr = np.asarray(vin_mV, dtype=float)
        i_arr = np.asarray(i_dc_uA, dtype=float)

        a, b, _ = self.popt_poly
        vin_V = vin_arr * 1e-3
        i_dc_A = i_arr * 1e-6

        # Piecewise df/dV: 0 below threshold, polynomial derivative above
        df_dV_mV = np.where(
            vin_arr < self.piecewise_threshold,
            0.0,
            2 * a * vin_arr + b
        )  # units: kHz/mV

        denom = self.vdd - vin_V
        with np.errstate(divide='ignore', invalid='ignore'):
            dG_dV = i_dc_A / (denom ** 2)  # S/V
            # Convert dG/dV from S/V to uS/mV:
            # (S/V) * (1e6 uS/S) * (1e-3 V/mV) = 1e3 uS/mV
            dG_dV_uS_per_mV = dG_dV * 1e3

            dG_df = dG_dV_uS_per_mV / df_dV_mV  # uS/kHz

        invalid = (
            (vin_arr < self.piecewise_threshold) |
            np.isnan(vin_arr) |
            np.isnan(i_arr) |
            (denom <= 0) |
            (df_dV_mV == 0)
        )
        dG_df = np.where(invalid, np.nan, dG_df)

        return dG_df.item() if np.isscalar(vin_mV) and np.isscalar(i_dc_uA) else dG_df

    def dG_df_curve(self, i_dc_range, vin_min=None, vin_max=None, n_points=200):
        """
        Returns G tolerance curves (dG/df) over a Vin sweep for each i_dc in i_dc_range.
        
        G tolerance quantifies the conductance uncertainty due to f_osc measurement error.
        Output shape: (len(i_dc_range), n_points)
        """
        if vin_min is None:
            vin_min = self.piecewise_threshold
        if vin_max is None:
            vin_max = max(np.max(self.vin_data), self.vdd_mV())

        vin_smooth = np.linspace(vin_min, vin_max, n_points)             
        i_dc_arr = np.asarray(i_dc_range, dtype=float)[:, None]           

        a, b, _ = self.popt_poly
        df_dV_mV = np.where(
            vin_smooth < self.piecewise_threshold,
            0.0,
            2 * a * vin_smooth + b
        )[None, :]                                                        

        vin_smooth_V = vin_smooth[None, :] * 1e-3                         
        denom = self.vdd - vin_smooth_V
        i_dc_A = i_dc_arr * 1e-6                                         

        with np.errstate(divide='ignore', invalid='ignore'):
            dG_dV = i_dc_A / (denom ** 2)                                 # S/V
            dG_dV_uS_per_mV = dG_dV * 1e3                                 # uS/mV
            sens = dG_dV_uS_per_mV / df_dV_mV                             # uS/kHz

        invalid = (vin_smooth[None, :] < self.piecewise_threshold) | (denom <= 0) | (df_dV_mV == 0)
        sens = np.where(invalid, np.nan, sens)

        return vin_smooth, sens
    
    def delta_f_kHz(self, fs_Hz=None, Ts_s=None, avg_window=1):
        """
        Estimate frequency resolution in kHz from counting over a window Ts.

        Parameters
        ----------
        fs_Hz : float or array-like, optional
            Sampling frequency in Hz.
        Ts_s : float or array-like, optional
            Measurement window in seconds.
        avg_window : int or float, optional
            Number of averaged windows. Improves resolution by sqrt(avg_window).

        Returns
        -------
        delta_f_kHz = 1 / (Ts_s * np.sqrt(avg_window)) : float or ndarray
            Estimated frequency uncertainty in kHz.
        """
        if Ts_s is None:
            if fs_Hz is None:
                return np.nan
            fs_Hz = np.asarray(fs_Hz, dtype=float)
            Ts_s = 1.0 / fs_Hz
        else:
            Ts_s = np.asarray(Ts_s, dtype=float)

        avg_window = np.asarray(avg_window, dtype=float)

        with np.errstate(divide='ignore', invalid='ignore'):
            delta_f_Hz = 1.0 / (Ts_s * np.sqrt(avg_window))
            delta_f_kHz = delta_f_Hz / 1e3

        invalid = (Ts_s <= 0) | (avg_window <= 0)
        delta_f_kHz = np.where(invalid, np.nan, delta_f_kHz)

        return delta_f_kHz.item() if np.isscalar(fs_Hz) or np.isscalar(Ts_s) else delta_f_kHz
    
    def delta_G(self, vin_mV, i_dc_uA, fs_Hz=None, Ts_s=None, avg_window=1):
        """
        Estimate conductance resolution/uncertainty in uS.

        Uses:
            DeltaG ≈ |dG/df| * Deltaf
        """
        dG_df = self.G_tolerance(vin_mV, i_dc_uA)  # uS/kHz
        delta_f_kHz = self.delta_f_kHz(fs_Hz=fs_Hz, Ts_s=Ts_s, avg_window=avg_window)

        with np.errstate(invalid='ignore'):
            delta_G_uS = np.abs(dG_df) * delta_f_kHz

        return delta_G_uS.item() if np.isscalar(vin_mV) and np.isscalar(i_dc_uA) else delta_G_uS
    
    def delta_G_vs_idc(self, vin_mV, i_dc_range, fs_Hz=None, Ts_s=None, avg_window=1):
        """
        Conductance resolution as a function of i_dc for a fixed Vin.
        """
        i_dc_arr = np.asarray(i_dc_range, dtype=float)
        return i_dc_arr, self.delta_G(vin_mV, i_dc_arr, fs_Hz=fs_Hz, Ts_s=Ts_s, avg_window=avg_window)

In [ ]:
class Power_Model:
    """
    Encapsulate power estimation from the iDAC and VCO.
    """
    def __init__(self, vco_model):
        self.vco_model = vco_model

    def power_from_idc(self, i_dc_uA):
        """
        Power in uW from i_dc in uA.

        Uses:
            P = I * V
        """
        i_arr = np.asarray(i_dc_uA, dtype=float)
        i_A = i_arr * 1e-6
        power_W = i_A * self.vco_model.vdd
        power_uW = power_W * 1e6
        return power_uW.item() if np.isscalar(i_dc_uA) else power_uW
        



In [3]:
# instantiate the model with our data
VCO_model = VCO_Model(
    vin_data=[225,250,275,300,330,340,360,380,400,420,440,460,480,500,
              520,540,560,580,600,620,640,660,680,700,720,740,760,780,800,820],
    fosc_data=[0,0,0,0,24,26,31,37,45,55,67,81,99,120,146,174,209,247,
               292,343,397,455,520,599,676,760,855,950,1050,1170]
)


Determined piecewise threshold: 330.00 mV


In [4]:
# ============================================================================
# f_osc statistical model from measured data
# ============================================================================

# Measured data from your graph
vin_data = VCO_model.vin_data
fosc_data = VCO_model.fosc_data

def plotter_fosc():
    """
    Plotter for f_osc piecewise polynomial model fitted to measured data.
    Uses VCO_Model instance for all calculations.
    
    Piecewise model:
        f_osc = 0                      if V_in < threshold
              = a*V_in^2 + b*V_in + c  otherwise
    """
    def plot_model():
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Generate smooth curve using model
        vin_smooth = np.linspace(225, 820, 300)
        fosc_fit = VCO_model.fosc_from_vin(vin_smooth)
        fosc_points = VCO_model.fosc_from_vin(vin_data)
        
        ax.scatter(vin_data, fosc_data, s=80, color='black', alpha=0.7, label='Measured data', zorder=5)
        ax.plot(vin_smooth, fosc_fit, 'purple', linewidth=2.5, label='Piecewise polynomial fit')
        
        # Show residuals
        residuals = fosc_data - fosc_points
        ax.scatter(vin_data, residuals, s=50, color='orange', alpha=0.6, label='Residuals')
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
                
        ax.set_xlabel('V_in (mV)', fontsize=12)
        ax.set_ylabel('f_osc (kHz) / Residuals', fontsize=12)
        ax.set_title('Piecewise Polynomial Model Fit', fontsize=13)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=10)
        ax.set_xlim(200, 850)
        
        plt.tight_layout()
        plt.show()
        
        # Print model statistics
        print(f"\n{'='*60}")
        print(f"Piecewise Polynomial Model")
        print(f"{'='*60}")
        print(f"Piecewise threshold: {VCO_model.piecewise_threshold:.2f} mV")
        print(f"\nPolynomial coefficients (active region only):")
        a, b, c = VCO_model.popt_poly
        print(f"  a = {a:.6f}")
        print(f"  b = {b:.4f}")
        print(f"  c = {c:.3f}")
        print(f"\nEquation: f_osc = {a:.6f} × V_in² + {b:.4f} × V_in + {c:.3f}")
        print(f"          (valid for V_in ≥ {VCO_model.piecewise_threshold:.2f} mV)")
        
        # Calculate fit statistics
        residuals = fosc_data - VCO_model.fosc_from_vin(vin_data)
        rmse = np.sqrt(np.mean(residuals**2))
        mae = np.mean(np.abs(residuals))
        print(f"\nFit Statistics (all data):")
        print(f"  RMSE: {rmse:.2f} kHz")
        print(f"  MAE:  {mae:.2f} kHz")
        
        # Statistics for active region only
        active_mask = vin_data >= VCO_model.piecewise_threshold
        residuals_active = residuals[active_mask]
        rmse_active = np.sqrt(np.mean(residuals_active**2))
        mae_active = np.mean(np.abs(residuals_active))
        print(f"\nFit Statistics (active region only, V_in ≥ {VCO_model.piecewise_threshold:.2f} mV):")
        print(f"  RMSE: {rmse_active:.2f} kHz")
        print(f"  MAE:  {mae_active:.2f} kHz")
        print(f"{'='*60}\n")
    
    interact(plot_model)

print("\n" + "="*70)
print("f_osc Statistical Model (fitted to measured data)")
print("="*70)
plotter_fosc()


f_osc Statistical Model (fitted to measured data)


interactive(children=(Output(),), _dom_classes=('widget-interact',))

In [5]:
# ============================================================================
# INTEGRATED PLOTTER: f_osc → V_in → G measurement with G tolerance analysis
# ============================================================================
"""
Measurement workflow with piecewise polynomial VCO model:
1. Measure f_osc from oscilloscope
2. Use piecewise model to find V_in from f_osc (f_osc=0 below threshold, polynomial above)
3. Calculate G = i_dc / (VDD - V_in) for measured V_in and applied i_dc
4. Assess G tolerance (dG/d(f_osc)) - how G varies with f_osc measurement error

The piecewise model captures the oscillator's threshold behavior:
- For V_in < {threshold}: f_osc = 0 (no oscillation)
- For V_in ≥ {threshold}: f_osc = a*V_in² + b*V_in + c (active region)
"""

def plotter_integrated_fosc_to_G():
    """
    Integrated plotter connecting f_osc measurement to G extraction
    with G tolerance analysis across different i_dc regimes.
    
    Uses the piecewise polynomial VCO model for all calculations.
    """    
    def plot_workflow(f_osc_measured_kHz, i_dc_uA):
        """
        Main plotting function for the integrated workflow
        """
        # Step 1: Convert measured f_osc to V_in using the piecewise oscillator model
        V_in_mV = VCO_model.vin_from_fosc(f_osc_measured_kHz)
        
        # Step 2: Calculate G from measured V_in and selected i_dc
        G_uS = VCO_model.conductance(V_in_mV, i_dc_uA)
        R_kohm = 1 / (G_uS / 1e6) / 1e3 if not np.isnan(G_uS) else np.nan
        
        # Step 3: Prepare G tolerance grid
        i_dc_range = np.linspace(0.1, 10, 100)  # 0.1 to 10 μA
        
        # G tolerance = dG/d(f_osc) along the chosen Vin for each i_dc
        # (quantifies G uncertainty due to f_osc measurement error)
        dG_dfs = [VCO_model.dG_df(V_in_mV, idc) for idc in i_dc_range]
        
        # Create comprehensive figure
        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
        
        # ===== Panel 1: f_osc Model with measurement point =====
        ax1 = fig.add_subplot(gs[0, 0])
        vin_plot = np.linspace(225, 820, 300)
        fosc_plot = VCO_model.fosc_from_vin(vin_plot)
        
        ax1.scatter(vin_data, fosc_data, s=60, color='black', alpha=0.6, label='Measured data', zorder=5)
        ax1.plot(vin_plot, fosc_plot, 'b-', linewidth=2.5, label='Piecewise polynomial model')
        
        
        if not np.isnan(V_in_mV) and 225 <= V_in_mV <= 820:
            ax1.plot(V_in_mV, f_osc_measured_kHz, 'r*', markersize=20, label=f'Measured: f={f_osc_measured_kHz:.0f} kHz', zorder=10)
            ax1.axvline(V_in_mV, color='r', linestyle='--', alpha=0.5)
        
        ax1.set_xlabel('V_in (mV)', fontsize=11)
        ax1.set_ylabel('f_osc (kHz)', fontsize=11)
        ax1.set_title('Step 1: f_osc Model & Measurement', fontsize=12, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        ax1.legend(fontsize=10)
        ax1.set_xlim(200, 850)
        
        # ===== Panel 2: V_in from f_osc measurement =====
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.text(0.5, 0.85, 'Step 2: Extract V_in from f_osc', transform=ax2.transAxes, 
                fontsize=12, fontweight='bold', ha='center')
        
        result_text = f"Measured f_osc: {f_osc_measured_kHz:.1f} kHz\n"
        result_text += f"Extracted V_in: {V_in_mV:.2f} mV\n"
        result_text += f"(using piecewise polynomial model)"
        result_text += f"\nThreshold: {VCO_model.piecewise_threshold:.1f} mV"
        
        if not np.isnan(V_in_mV):
            result_text += f"\n\n✓ Valid range"
        else:
            result_text += f"\n\n✗ Outside valid range!"
        
        ax2.text(0.5, 0.45, result_text, transform=ax2.transAxes, fontsize=13,
                ha='center', va='center', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
                family='monospace')
        ax2.axis('off')
        
        # ===== Panel 3: G calculation for selected i_dc =====
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.text(0.5, 0.85, 'Step 3: Calculate G for given i_dc', transform=ax3.transAxes, 
                fontsize=12, fontweight='bold', ha='center')
        
        calc_text = f"i_dc: {i_dc_uA:.2f} μA\n"
        calc_text += f"V_in: {V_in_mV:.2f} mV\n"
        calc_text += f"─────────────────\n"
        calc_text += f"G = i_dc / (VDD - V_in)\n"
        
        if not np.isnan(G_uS):
            calc_text += f"\nG: {G_uS:.4f} uS\n"
            calc_text += f"R: {R_kohm:.2f} kΩ"
        else:
            calc_text += f"\n✗ Cannot calculate"
        
        ax3.text(0.5, 0.45, calc_text, transform=ax3.transAxes, fontsize=12,
                ha='center', va='center', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8),
                family='monospace')
        ax3.axis('off')
        
        # ===== Panel 4: V_in vs i_dc (for reference) =====
        i_dc_range_A = i_dc_range * 1e-6  # convert to A for plotting

        ax4 = fig.add_subplot(gs[1, 1])
        ax4.plot(i_dc_range, (i_dc_range_A / ((VCO_model.vdd - V_in_mV * 1e-3)) * 1e6) if not np.isnan(V_in_mV) else [], 
         'g-', linewidth=2.5, label=f'V_in = {V_in_mV:.1f} mV')
        if not np.isnan(V_in_mV):
            ax4.plot(i_dc_uA, G_uS, 'r*', markersize=20, label=f'Selected i_dc = {i_dc_uA:.2f} μA', zorder=5)
        
        ax4.set_xlabel('i_dc (μA)', fontsize=11)
        ax4.set_ylabel('G (uS)', fontsize=11)
        ax4.set_title('G vs i_dc at fixed V_in', fontsize=12, fontweight='bold')
        ax4.grid(True, alpha=0.3)
        ax4.legend(fontsize=10)
        # ===== Panel 5: G Tolerance analysis - dG/d(fosc) =====
        ax5 = fig.add_subplot(gs[2, :])
        # Calculate delta G/d(f_osc) at the measured f_osc point for different i_dc values
        dG_dfs = [VCO_model.dG_df(V_in_mV, idc) for idc in i_dc_range]
        ax5.plot(i_dc_range, dG_dfs, 'purple', linewidth=3, label='G Tolerance: dG/d(f_osc) [uS/kHz]')
        ax5.plot(i_dc_uA, dG_dfs[np.argmin(np.abs(i_dc_range - i_dc_uA))], 'r*', 
                markersize=20, label=f'Current setting: {i_dc_uA:.2f} μA', zorder=5)
        ax5.fill_between(i_dc_range, dG_dfs, alpha=0.3, color='purple')
        ax5.set_ylabel('G Tolerance [uS/kHz]', fontsize=11)
        ax5.set_title('Step 4: G Tolerance Analysis - How much does G vary with f_osc measurement error?', 
                     fontsize=12, fontweight='bold')
        ax5.grid(True, alpha=0.3)
        ax5.legend(fontsize=10)
        
        plt.suptitle('Integrated Measurement Workflow: f_osc → V_in → G', 
                    fontsize=14, fontweight='bold', y=0.995)
        plt.show()
        
        # Print detailed analysis
        print("\n" + "="*70)
        print("MEASUREMENT WORKFLOW ANALYSIS")
        print("="*70)
        print(f"\n1. MEASUREMENT INPUT:")
        print(f"   f_osc (measured):  {f_osc_measured_kHz:.1f} kHz")
        print(f"   Model used:        Piecewise Polynomial")
        print(f"   Threshold:         {VCO_model.piecewise_threshold:.2f} mV")
        
        print(f"\n2. EXTRACTED V_IN:")
        if not np.isnan(V_in_mV):
            print(f"   V_in:              {V_in_mV:.3f} mV")
        else:
            print(f"   V_in:              ✗ OUT OF RANGE")
        
        print(f"\n3. CONDUCTANCE CALCULATION:")
        print(f"   i_dc (applied):    {i_dc_uA:.3f} μA")
        if not np.isnan(G_uS):
            print(f"   G:                 {G_uS:.6f} uS")
            print(f"   R:                 {R_kohm:.4f} kΩ")
        else:
            print(f"   G:                 ✗ CANNOT CALCULATE")
        # G tolerance analysis
        print(f"\n4. G TOLERANCE ANALYSIS:")
        current_dG_df = dG_dfs[np.argmin(np.abs(i_dc_range - i_dc_uA))]
        print(f"   dG/d(f_osc):       {current_dG_df:.6f} uS/kHz")
        print(f"   Interpretation:    1 kHz error in f_osc → {current_dG_df:.6f} uS error in G")
        
        max_tolerance_idx = np.argmax(dG_dfs)
        print(f"\n   Lowest resolution at: {i_dc_range[max_tolerance_idx]:.2f} μA ({dG_dfs[max_tolerance_idx]:.6f} uS/kHz)")
        min_tolerance_idx = np.argmin(np.array(dG_dfs)[np.array(dG_dfs) > 0]) if any(t > 0 for t in dG_dfs) else 0
        if dG_dfs[min_tolerance_idx] > 0:
            print(f"   Highest resolution at: {i_dc_range[min_tolerance_idx]:.2f} μA ({dG_dfs[min_tolerance_idx]:.6f} uS/kHz)")
        
        print("\n" + "="*70 + "\n")
        
    
    # Create interactive controls
    f_osc_slider = FloatSlider(
        value=820, 
        min=0, 
        max=1200, 
        step=10,
        description='f_osc (kHz):', 
        continuous_update=True
    )
    
    i_dc_slider = FloatSlider(
        value=1.0, 
        min=0.1, 
        max=10, 
        step=0.1,
        description='i_dc (μA):', 
        continuous_update=True
    )
    
    interact(plot_workflow, 
             f_osc_measured_kHz=f_osc_slider,
             i_dc_uA=i_dc_slider)


print("\n" + "="*70)
print("INTEGRATED PLOTTER: f_osc → G measurement with G tolerance analysis")
print("="*70)
plotter_integrated_fosc_to_G()




INTEGRATED PLOTTER: f_osc → G measurement with G tolerance analysis


interactive(children=(FloatSlider(value=820.0, description='f_osc (kHz):', max=1200.0, step=10.0), FloatSlider…